# 21 Sep

## Idea here is to make 4 scripts that will be capable of taking bulk csv in a normalized format that will then be able to run and automatically make data into a wanddb friendly format and standardise it


## Individual compound normaliser
Expected inputs:
- bulk inchi list from wand (required): filepath
- name : string
- compound_source : string
- smiles/inchi/iupac (at least 1 required) : string
- 

In [ ]:
import pandas as pd
from rdkit import Chem
import nispo
from Pyopsin import OpSin

wand_inchis_path = ''
name = ''
source = ''
smiles = ''
inchi = ''
iupac = ''



## bulk compound normaliser
not going to make it too interoperable just yet but for now I will make it so that it can use a file with inchis

- Expected inputs: 
- bulk inchi list from wand with names, synonyms, iupac and pubchemids (required): filepath
- compound_csv (column names as stated): 
- 'name' : string
- 'compound_source' : string
- 'smiles'/'inchi'/'iupac' (at least 1 required) : string

In [ ]:
# wand compounds expected headings ['compound_id','inchi','name','synonyms','iupac']
# new compounds expected headings ['id','new_id','inchi','n_name','n_synonyms','n_iupac'] 
# (new_id column must be named with source then _id i.e. lotus_id, wiki_id, can have multiple)
# all multi fields must be '|' seperated if have multi values
#//////////////////////////////////////////////////////inputs
wand_inchis_path = ''
new_compounds_path = ''
staging_table=''
error_output_path = ''
output_path = ''
report_path = ''
has_new_ids = True
sql_commands = ''
#//////////////////////////////////////////////////////imports
import pandas as pd
from rdkit import Chem
import time
import pubchempy as pcp
from nispo import mol_to_iupac
#//////////////////////////////////////////////////////data
wand_compounds = pd.read_csv(wand_inchis_path)
new_compounds = pd.read_csv(new_compounds_path)
origional_size = new_compounds.shape[0]
errors = pd.DataFrame(columns = ['id','type','error_msg'])
print('Files loaded', flush = True)
#//////////////////////////////////////////////////////functions
def get_smiles2D(inchi):
    global errors

    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            c_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
            return c_smiles
        else:
            err = [inchi,'inchi','Could not convert inchi to 2D smiles']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_smiles3D(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            i_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
            return i_smiles
        else:
            err = [inchi,'inchi','Could not convert inchi to isomericsmiles']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_cid(input,type):
    global errors
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                err = [input,type,'Does not appear to be in pubchem with this inchi']
                errors.loc[len(errors)] = err
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(0.2)
            else:
                err = [input,type,str(e)]
                errors.loc[len(errors)] = err
                return None
    err = [input,type,'retry not sucessful']
    errors.loc[len(errors)] = err
    return None

def get_inchikey(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)
        if mol:
            key = Chem.MolToInchiKey(mol)
            return key
        else:
            err = [inchi,'inchi','Could not convert inchi to inchi key']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_iupac(inchi):
    global errors
    try:
        mol = Chem.MolFromInchi(inchi,treatWarningAsError=True)
        if mol:
            iupac = mol_to_iupac(mol)
            if pd.notna(iupac):
                return iupac
            else:
                err = [inchi, 'inchi', 'Could not generate IUPAC name']
                errors.loc[len(errors)] = err
                return None
        else:
          err = [inchi,'inchi','Could not convert inchi to iupac']
          errors.loc[len(errors)] = err
          return None  
    except Exception as e:
        err = [inchi,'inchi',str(e)]
        errors.loc[len(errors)] = err
        return None 

def synonym_deduplicator(old,new):
    old = '' if pd.isna(old) else str(old)
    new = '' if pd.isna(new) else str(new)
    old = old.split('|')
    new = new.split('|')
    old.remove('[]')
    new.remove('[]')
    old = [ol.strip() for ol in old if ol.strip()]
    new = [ne.strip() for ne in new if ne.strip()]

    old_set = set(old)
    old_set = {ol.lower() for ol in old_set}
    for ne in new:
        if ne and ne.lower() not in old_set:
            old.append(ne)
            old_set.add(ne.lower())
    return '|'.join(old)

def name_syn_remover(name,synonyms):
    name = '' if pd.isna(name) else str(name)
    synonyms = '' if pd.isna(synonyms) else str(synonyms)
    name = name.strip()
    synonyms = synonyms.split('|')
    synonyms = [s.strip() for s in synonyms if s.strip()]
    synonyms = [s for s in synonyms if s.lower() != name.lower()]
    return '|'.join(synonyms)

def cid_with_progress(series,function):
    tot = len(series)
    count = 0
    def wrapped(value):
        nonlocal count
        count += 1
        if count % 100==0 or count == tot or count == 1:
            print(f'CID progress: {count} of {tot} at {count/tot*100:.2f}%', flush=True)
        result = function(value)
        return result
    return series.apply(wrapped)
#//////////////////////////////////////////////////////new compounds
                        #//////////////////////////////check if inchis are missing or if are already in the DB and remove from new add to err and add as new metadata
wand_set = set(wand_compounds['inchi'].tolist())

if new_compounds['inchi'].isna().any():
    missing_inchi = pd.DataFrame(columns=['id','type','error_msg'])
    missing_inchi['id'] = new_compounds['id'][new_compounds['inchi'].isna()]
    missing_inchi['type'] = 'inchi'
    missing_inchi['error_msg'] = 'missing an inchi'
    errors = pd.concat([errors,missing_inchi], ignore_index=True)
new_compounds.dropna(subset=['inchi'], inplace=True)

in_db = pd.DataFrame(columns=['id','type','error_msg'])
in_db['id'] = new_compounds[new_compounds['inchi'].isin(wand_set)]['inchi'].to_list()
in_db['type'] = 'inchi'
in_db['error_msg'] = 'inchi already in DB'
errors = pd.concat([errors,in_db], ignore_index=True)
compounds_already_in_db = new_compounds[new_compounds['inchi'].isin(wand_set)].copy()
new_compounds = new_compounds[~new_compounds['inchi'].isin(wand_set)].copy()

print('Seperated onld and new', flush=True)
                        #//////////////////////////////add standardized smiles
print('Standardising smiles', flush=True)
new_compounds['canonical_smiles'] = new_compounds['inchi'].apply(get_smiles2D)
new_compounds['isomeric_smiles'] = new_compounds['inchi'].apply(get_smiles3D)
print('Standardised smiles', flush=True)
                        #//////////////////////////////add cid
print('Adding cid', flush=True)
new_compounds['cid'] = cid_with_progress(new_compounds['inchi'],lambda inchi: get_cid(inchi, 'inchi'))
print('cids added', flush=True)
                        #//////////////////////////////Add iupac
print('Adding iupac', flush=True)
new_compounds['n_iupac'] = new_compounds['n_iupac'].fillna(new_compounds['inchi'].apply(get_iupac))
print('iupac added', flush=True)
                        #//////////////////////////////Add inchikey
print('Adding inchikey', flush=True)
new_compounds['inchi_key'] = new_compounds['inchi'].apply(get_inchikey)
print('DB formatting', flush=True)
                        #//////////////////////////////wand_ids
start_id = pd.to_numeric(wand_compounds['compound_id']).max()+1
new_compounds['compound_id'] = range(start_id, start_id+len(new_compounds))
                        #//////////////////////////////correct columns
new_compounds['dois'] = None
new_compounds['collections'] = None
new_compounds['coconut_id'] = None
new_compounds['cas'] = None
new_compounds = new_compounds.rename(columns={'n_name': 'name','n_synonyms': 'synonyms','n_iupac': 'iupac'})
new_compounds = new_compounds[['compound_id', 'name', 'inchi', 'inchi_key', 'canonical_smiles', 'isomeric_smiles', 'iupac', 'synonyms', 'collections', 'dois', 'coconut_id', 'cid', 'cas']]
print('New Compounds formatted and done', flush=True)
#//////////////////////////////////////////////////////update existing
print('Updating existing', flush=True)
                        #//////////////////////////////making working df for eventaul staging table
already_compounds = set(compounds_already_in_db['inchi'].to_list())
staging_df = wand_compounds.loc[wand_compounds['inchi'].isin(already_compounds)].copy()
                        #//////////////////////////////updating names,iupac and synonyms
staging_df = staging_df.merge(compounds_already_in_db, how='left',on='inchi')

staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['n_synonyms']),axis=1)
staging_df.drop(columns =['n_synonyms'], inplace=True)

staging_df['name'] = staging_df['name'].fillna(staging_df['n_name'])
staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['n_name']), axis=1)

staging_df['synonyms'] = staging_df.apply(lambda row: name_syn_remover(row['name'],row['synonyms']), axis=1)
staging_df.drop(columns = ['n_name'], inplace=True)

staging_df['iupac'] = staging_df['iupac'].fillna(staging_df['n_iupac'])
staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['iupac']), axis=1)
staging_df.drop(columns = ['n_iupac'], inplace=True)
print('Updated existing', flush=True)
#//////////////////////////////////////////////////////report
print('Generating report', flush=True)
new_size = new_compounds.shape[0]
errors_size = errors.shape[0]

items_removed = origional_size - new_size

error_types = errors['error_msg'].value_counts(dropna=False)

item = f"Compounds in starting file: {origional_size}\n\
         New compounds standardised to add to the database: {new_size}\n\
         Removed compounds: {items_removed}\n\
         Errors: {errors_size}\n\
         Error types:\n\
         {error_types}\n\
         "

with open(report_path, 'w') as f:
    f.write(item)
#//////////////////////////////////////////////////////output
print('Saving', flush=True)
new_compounds.to_csv(output_path, index=False) 
staging_df.to_csv(staging_table, index=False)
errors.to_csv(error_output_path, index=False)

# This is a CID getter

In [ ]:
#!/bin/python3
# column names needed:['inchi','inchi_key','smiles']
#module load python-3.9.5
#module load pubchempy
#//////////////////////////////////////////////////input
compounds_in = ''
err_out = ''
results_out = ''
#//////////////////////////////////////////////////imports
import time
import pandas as pd
import pubchempy as pcp
#//////////////////////////////////////////////////functions
def get_cid(input,type):
    if pd.isna(input):
        return f'not input ({type})'
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return str(e)
    return 'retry not sucessful'
       
def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out
#///////////////////////////////////////////////main
df = pd.read_csv(compounds_in)

df['cid'] = None
err = []
count = 1
length = len(df)

for idx, row in df.iterrows():
    inchi = row['inchi']
    inchi_key = row['inchi_key']
    smiles = row['smiles']

    cid = get_cid(inchi,'inchi')
    prog = progress_bar(count, length)

    if isinstance(cid,int):
        df.at[idx, 'cid'] = cid
    else:
        err.append([inchi,cid])
        cid = get_cid(smiles,'smiles')
        if isinstance(cid,int):
            df.at[idx, 'cid'] = cid
        else:
            err.append([inchi,cid])
            cid = get_cid(inchi_key,'inchikey')
            if isinstance(cid,int):
                df.at[idx, 'cid'] = cid
            else:
                err.append([inchi,cid])
    time.sleep(0.2)
    count += 1

    if count % 1000 == 0 or count == 1 or count == len(df):
        df.to_csv(results_out, index=False)
        error_df = pd.DataFrame(err, columns=['inchi', 'error'])
        error_df.to_csv(err_out,index=False)
    print(prog)

#////////////////////////////////////////////////////outputs
df.to_csv(results_out,index=False)
error_df = pd.DataFrame(err, columns=['inchi', 'error'])
error_df.to_csv(err_out,index=False)